In [0]:
import sys
sys.path.append("/Workspace/Shared/databricks-practice/utils/")

import pyspark.sql.functions as F

from functions import *

In [0]:
df = spark.read.table("databricks_practice.bronze.bronze_table")
display(df.select("sku").distinct())

In [0]:
display(df.count())

In [0]:
display(df.select("catalog_sku_id").distinct())

In [0]:
bronze_table = "databricks_practice.bronze.bronze_table"
silver_table = "databricks_practice.silver.dim_sku"
partition_col = "dt_ingestion_partition"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricks_practice.silver.dim_sku (
  sku_sk BIGINT GENERATED ALWAYS AS IDENTITY,
  sku STRING,
  catalog_sku_id STRING,
  stock_keeping_unit_id STRING,
  product_id STRING,
  dt_ingestion_partition DATE,
  update_ts TIMESTAMP
) USING DELTA;


In [0]:
df_new = load_bronze_new_partitions(spark, bronze_table, silver_table, partition_col)

In [0]:
df_sku_silver = clean_sku(df_new)

In [0]:
df_dim_sku = (df_sku_silver) \
    .select("sku", "catalog_sku_id", "stock_keeping_unit_id", "product_id", "dt_ingestion_partition") \
    .withColumn("dt_ingestion_partition", F.col("dt_ingestion_partition").cast("date")) \
    .dropDuplicates() \
    .dropna() 
  #  .withColumn("market_sk", F.sha2(F.concat_ws("||", F.col("country_code")), 256)

In [0]:
df_dim_sku.createOrReplaceTempView("df_dim_sku")

In [0]:
%sql
MERGE INTO databricks_practice.silver.dim_sku AS silver
USING df_dim_sku AS bronze
ON silver.sku = bronze.sku

WHEN MATCHED THEN UPDATE SET
  silver.catalog_sku_id = bronze.catalog_sku_id,
  silver.stock_keeping_unit_id = bronze.stock_keeping_unit_id,
  silver.product_id = bronze.product_id,
  silver.dt_ingestion_partition = bronze.dt_ingestion_partition,
  silver.update_ts = current_timestamp()

WHEN NOT MATCHED THEN INSERT (
  sku, catalog_sku_id, stock_keeping_unit_id, product_id, dt_ingestion_partition, update_ts
)
VALUES (
  bronze.sku, bronze.catalog_sku_id, bronze.stock_keeping_unit_id, bronze.product_id, bronze.dt_ingestion_partition, current_timestamp()
);

In [0]:
# Teste processamento overwrite quantos segundos? Overwrite abaixo
# - MERGE INTO 15 segundos
# - OVERWRITE 10 segundos

In [0]:
df_dim_sku.createOrReplaceTempView("dim_sku_teste")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricks_practice.silver.dim_sku_teste (
  sku_sk BIGINT GENERATED ALWAYS AS IDENTITY,
  sku STRING,
  catalog_sku_id STRING,
  stock_keeping_unit_id STRING,
  product_id STRING,
  dt_ingestion_partition DATE,
  update_ts TIMESTAMP
) USING DELTA;


In [0]:
df_dim_sku = df_dim_sku.withColumn("update_ts", F.current_timestamp())

df_dim_sku.write \
    .mode("overwrite") \
    .saveAsTable("databricks_practice.silver.dim_sku_teste")

In [0]:
%sql
SELECT * FROM databricks_practice.silver.dim_sku_teste LIMIT 10

In [0]:
# Antes 1447795 linhas

In [0]:
df_sku_silver_limpo = (df_sku_silver) \
    .select("sku", "catalog_sku_id", "stock_keeping_unit_id", "product_id")

df_sku_silver_limpo = df.withColumn("chave", F.concat_ws("||", *df_sku_silver_limpo.columns))

In [0]:
# Trazer mais alguma coluna de produto para ver se é unica essa chave da sku

In [0]:
display(df_sku_silver_limpo.groupby("chave").count().filter(F.col("count") > 1))

In [0]:
display(df_sku_silver_limpo.groupby("chave").count().filter(F.col("count") > 1).agg(F.sum("count").alias("sum_count")))

In [0]:
display(df_sku_silver_limpo.groupby("chave").count().filter(F.col("count") >= 1).agg(F.count("chave").alias("total_chaves_com_mais_de_1_ocorrencia")))

In [0]:
display(df_sku_silver_limpo.filter(F.col("chave").isNull()))

In [0]:
display(df_sku_silver_limpo.filter(F.col("sku").isNull()))

In [0]:
# Talvez fazer uma regra para concatenar e verificar se os counts sao maiores que 1 no distinct (Verificar se um sku esta em diferentes catalogs_sku_id e por ai vai)